# 07 — GÉNÉRATION : l'IA parle djembé 🥁

Première version du but final. À partir des **partitions symboliques** (notebook 06) :
1. On apprend la **grammaire du soliste** : un modèle de Markov `P(frappe | position dans le cycle, frappe précédente)`, par pièce.
2. On **génère** des cycles de solo inédits.
3. On **sonifie** avec de vraies frappes découpées du corpus (banque d'échantillons), accompagnement dunun + J2 tiré des profils réels de la pièce.

→ On peut **écouter l'IA jouer** du Manjanin, du Maraka ou du Woloso.

## 0. Imports & données

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import djembe_utils as dj

strokes = pd.read_csv('output/all_jembe_strokes_typed.csv')   # frappes typées (06)
score   = pd.read_csv('output/partitions.csv')                # partitions (06)
c = dj.list_corpus()
VOC = ['-','b','t','s']
print(len(score), 'subdivisions de partition |', len(strokes), 'frappes typées')

## 1. Choisir la pièce & apprendre la grammaire (Markov)
Le modèle : à chaque **position** du cycle (1–12) et selon la **frappe précédente**, quelle est la probabilité de jouer `-`, `b`, `t`, `s` ? (comptage + lissage)

In [ ]:
PIECE = 'Woloso'    # <- 'Manjanin', 'Maraka', 'Woloso'
SEED  = 42          # <- change pour une improvisation différente

sub = score[(score['piece']==PIECE) & (score['J1']!='?')].reset_index(drop=True)
counts, prev = {}, '-'
for _, r in sub.iterrows():
    key = (r['SD'], prev)
    counts.setdefault(key, {v: 0.5 for v in VOC})
    counts[key][r['J1']] += 1
    prev = r['J1']
print(f'Grammaire {PIECE} : {len(counts)} contextes (position × frappe précédente)')

## 2. Générer des cycles de solo

In [ ]:
rng = np.random.RandomState(SEED)
def gen_cycles(n=8):
    seq, prev = [], '-'
    for _ in range(n):
        for sd in range(1, 13):
            d = counts.get((sd, prev), {v: 1 for v in VOC})
            syms = list(d); w = np.array([d[s] for s in syms], float)
            s = rng.choice(syms, p=w/w.sum())
            seq.append((sd, s)); prev = s
    return seq

gen = gen_cycles(8)
print('Solo généré (8 cycles, 1 caractère = 1 subdivision) :')
txt = ''.join(s for _, s in gen)
for i in range(0, len(txt), 12):
    print(' cycle', i//12+1, ':', txt[i:i+12])

## 3. Sonification : rejouer avec de VRAIES frappes
- **Soliste** : banque de vraies frappes b/t/s découpées des pistes J1 de la pièce.
- **Accompagnement** : dunun + J2 joués selon leurs **profils métriques réels**.
- Tempo réglable via `SUBDUR` (durée d'une subdivision).

In [ ]:
SUBDUR = 0.14   # s / subdivision (plus petit = plus rapide)

def prof_mean(code):
    ps = []
    for t in c.loc[(c['piece']==PIECE) & (c['code']==code), 'take'].unique():
        p = dj.subdivision_profile(PIECE, int(t), code)
        if p is not None: ps.append(p.values)
    return np.mean(ps, axis=0)
dprof, j2prof = prof_mean('D1'), prof_mean('J2')

def bank_j1(stype, n=6):
    s = strokes[(strokes['stroke']==stype) & (strokes['code']=='J1') & (strokes['piece']==PIECE)]
    s = s.nlargest(200, 'rms').sample(n, random_state=0)
    clips = []
    for _, r in s.iterrows():
        y = dj.load_track(r['piece'], int(r['take']), r['code'])
        a = int(r['time']*dj.SR); clip = y[a:a+int(0.22*dj.SR)].copy()
        clip[-400:] *= np.linspace(1, 0, 400)
        clips.append(clip)
    return clips
BJ1 = {'b': bank_j1('bass'), 't': bank_j1('tone'), 's': bank_j1('slap')}
yD = dj.load_track(PIECE, 1, 'D1'); onD = dj.load_onsets(PIECE, 1, 'D1')['Time'].values
BD  = [yD[int(t*dj.SR):int(t*dj.SR)+int(0.25*dj.SR)] for t in onD[20:26]]
yJ2 = dj.load_track(PIECE, 1, 'J2'); onJ2 = dj.load_onsets(PIECE, 1, 'J2')['Time'].values
BJ2 = [yJ2[int(t*dj.SR):int(t*dj.SR)+int(0.20*dj.SR)] for t in onJ2[20:26]]

out = np.zeros(int(len(gen)*SUBDUR*dj.SR + dj.SR))
for i, (sd, s) in enumerate(gen):
    t0 = int(i*SUBDUR*dj.SR)
    if rng.rand() < dprof[sd-1]:
        cl = BD[rng.randint(len(BD))];  out[t0:t0+len(cl)] += cl*0.9
    if rng.rand() < j2prof[sd-1]:
        cl = BJ2[rng.randint(len(BJ2))]; out[t0:t0+len(cl)] += cl*0.7
    if s != '-':
        cl = BJ1[s][rng.randint(len(BJ1[s]))]; out[t0:t0+len(cl)] += cl
out /= (np.abs(out).max() + 1e-9)
sf.write(f'output/generation_{PIECE}.wav', out, dj.SR)
print(f'>>> {PIECE} généré par l\'IA ({len(out)/dj.SR:.1f} s) :')
display(Audio(out, rate=dj.SR))

## 4. Comparer avec l'original
Un extrait réel de la même pièce, pour juger à l'oreille si la génération « sonne » Woloso.

In [ ]:
mix, _ = dj.build_mix(PIECE, 1)
ex = mix[int(30*dj.SR):int(44*dj.SR)]
print(f'>>> {PIECE}_1 réel (30–44 s) :'); display(Audio(ex, rate=dj.SR))

## Bilan & au-delà
- **v1 fonctionnelle** : grammaire Markov (position × frappe précédente) + sonification par vraies frappes → du djembé généré audible, dans le style de chaque pièce.
- Limites assumées : grille **isochrone** (le vrai jeu a un micro-timing non-isochrone → cf. `08_microtiming` pour l'injecter), pas de structure longue (phrases, chauffés, signaux) — un modèle d'ordre supérieur ou un petit transformer serait l'étape suivante.
- Change `PIECE` et `SEED` pour improviser autre chose.